# LeetCode 2651 — Calculate Delayed Arrival Time bằng Python

Link đề: https://leetcode.com/problems/calculate-delayed-arrival-time/ · Độ khó: **Easy** · Chủ đề: Math

Skeleton LeetCode:

```python
class Solution:
    def findDelayedArrivalTime(self, arrivalTime: int, delayedTime: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu bài toán **cộng giờ trên đồng hồ 24 giờ**: sau `23` giờ là `0` giờ (nửa đêm), nên phép cộng "quay vòng" thay vì tăng mãi.
2. Hiểu **số học đồng dư (modular arithmetic)** ở mức nhập môn: phép `%` (chia lấy số dư) biến mọi số nguyên về khoảng `0..23` nếu ta lấy `% 24`.
3. Phân biệt `%` (số dư) và `//` (thương nguyên), biết `divmod`, và biết Python xử lý `%` với **số âm** khác nhiều ngôn ngữ khác (kết quả luôn không âm khi chia cho số dương).
4. Viết 5 cách: mô phỏng từng giờ, `if` trừ `24`, phép `%`, dùng `datetime.timedelta`, và xoay vòng bằng `collections.deque.rotate`; so sánh và đối chiếu bằng cách thử **toàn bộ** cặp đầu vào hợp lệ.
5. Tránh các lỗi: quên quay vòng (trả `24`, `25`...), dùng `% 23`, chỉ trừ `24` bằng `if total > 24` (lệch một đơn vị), và nhầm `0` với `24`.

## 2. Đề bài & ví dụ

Cho số nguyên dương `arrivalTime` là **giờ tàu dự kiến đến** ga và số nguyên dương `delayedTime` là **số giờ tàu bị trễ**. Hãy trả về **giờ tàu thực sự đến ga**.

Chú ý: thời gian dùng **định dạng 24 giờ**, tức giờ nằm trong khoảng `0..23`; sau `23` giờ là `0` giờ (nửa đêm).

| Ví dụ | arrivalTime, delayedTime | Output | Giải thích |
|---|---|---|---|
| 1 | `15`, `5` | `20` | Dự kiến 15 giờ, trễ 5 giờ ⇒ 20 giờ. |
| 2 | `13`, `11` | `0` | `13 + 11 = 24`, mà 24 giờ chính là `0` giờ của ngày hôm sau. |

Constraints:

- `1 <= arrivalTime < 24`
- `1 <= delayedTime <= 24`

Hệ quả: tổng `arrivalTime + delayedTime` nằm trong khoảng `2..47` nên **tối đa chỉ quay vòng một lần**.

In [1]:
for a, d in [(15, 5), (13, 11)]:
    print(f"đến dự kiến {a}h, trễ {d}h -> cộng thẳng được {a + d}")

đến dự kiến 15h, trễ 5h -> cộng thẳng được 20
đến dự kiến 13h, trễ 11h -> cộng thẳng được 24


## 3. Kiến thức nền

### 3.1. Đồng hồ là một vòng tròn

Đồng hồ 24 giờ có các vạch `0, 1, 2, ..., 23`, rồi lại quay về `0`. Nếu ta cứ cộng bình thường, `23 + 1 = 24` nhưng đồng hồ phải chỉ `0`. Việc "quay vòng" này gọi là phép tính **modulo 24**.

Cách nghĩ: một ngày có 24 giờ; mỗi lần cộng đủ 24 giờ ta quay về chỗ cũ. Vậy `24 ≡ 0`, `25 ≡ 1`, `30 ≡ 6`, `47 ≡ 23` (mod 24).

In [2]:
for total in [20, 23, 24, 25, 30, 47, 48]:
    print(f"tổng {total:2} giờ -> đồng hồ chỉ {total % 24}")

tổng 20 giờ -> đồng hồ chỉ 20
tổng 23 giờ -> đồng hồ chỉ 23
tổng 24 giờ -> đồng hồ chỉ 0
tổng 25 giờ -> đồng hồ chỉ 1
tổng 30 giờ -> đồng hồ chỉ 6
tổng 47 giờ -> đồng hồ chỉ 23
tổng 48 giờ -> đồng hồ chỉ 0


### 3.2. Phép chia lấy số dư `%` và chia nguyên `//`

- `a // b`: **thương nguyên** (làm tròn xuống). `47 // 24 = 1` (qua một ngày).
- `a % b`: **số dư**. `47 % 24 = 23` (còn lại 23 giờ).
- Luôn có `a == (a // b) * b + a % b`. Ví dụ: `47 = 1 × 24 + 23`.

Số dư của phép chia cho `24` luôn thuộc `0..23` — đúng tập giá trị hợp lệ của giờ.

In [3]:
a, b = 47, 24
print(a // b, a % b)                          # 1 23
print((a // b) * b + a % b == a)              # True
print(divmod(a, b))                           # (1, 23): lấy cả thương và dư một lúc

1 23
True
(1, 23)


### 3.3. `%` với số âm trong Python

Khi chia cho số **dương**, Python luôn trả số dư **không âm**: `-1 % 24 == 23` (lùi một giờ từ `0` giờ thì ra `23` giờ — hợp lý với đồng hồ!). Nhiều ngôn ngữ khác (C, Java) có thể trả `-1`; ở LeetCode với Python bạn không phải lo, nhưng nếu dùng C++/Java phải viết `((x % 24) + 24) % 24`.

In [4]:
print(-1 % 24)       # 23
print(-25 % 24)      # 23
print(-24 % 24)      # 0
print((3 - 5) % 24)  # 22: 3 giờ lùi 5 giờ = 22 giờ ngày hôm trước

23
23
0
22


### 3.4. `if` / `else` kiểu biểu thức (toán tử điều kiện)

`A if điều_kiện else B` cho một giá trị, tiện cho các phép tính ngắn.

```python
x = total - 24 if total >= 24 else total
```

In [5]:
for total in [20, 23, 24, 25]:
    x = total - 24 if total >= 24 else total
    print(total, "->", x)

20 -> 20
23 -> 23
24 -> 0
25 -> 1


### 3.5. `datetime.timedelta` và `collections.deque.rotate`

- `datetime.timedelta(hours=5)` là một khoảng thời gian 5 giờ; cộng vào một `datetime` ta được thời điểm mới; thuộc tính `.hour` lấy giờ.
- `deque(range(24))` là hàng đợi hai đầu chứa `0..23`; `rotate(-k)` **xoay trái** `k` bước (phần tử đầu chuyển xuống cuối). Hàng đợi sau khi xoay trái `k` bước có phần tử thứ `i` là `(i + k) % 24`.

Hai công cụ này là cách "nhìn khác" về cùng một phép toán; ta sẽ dùng chúng làm cách thứ 4 và 5.

In [6]:
from datetime import datetime, timedelta
from collections import deque

t = datetime(2000, 1, 1, 13) + timedelta(hours=11)
print(t, "-> giờ", t.hour)              # 2000-01-02 00:00:00 -> giờ 0

dq = deque(range(5))                     # deque([0, 1, 2, 3, 4])
dq.rotate(-2)                            # xoay trái 2 bước
print(dq)                                # deque([2, 3, 4, 0, 1])

2000-01-02 00:00:00 -> giờ 0
deque([2, 3, 4, 0, 1])


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Cách người bình thường nghĩ: "tàu dự kiến 13 giờ, trễ 11 giờ. 13 + 11 = 24. Mà 24 giờ là nửa đêm, tức 0 giờ". Công thức tổng quát chỉ là:

> giờ thực = (giờ dự kiến + số giờ trễ) **mod 24**

Tính tay vài trường hợp:

- `15 + 5 = 20 < 24` ⇒ không quay vòng ⇒ `20`.
- `13 + 11 = 24` ⇒ vừa tròn một ngày ⇒ `0`.
- `23 + 24 = 47` ⇒ `47 - 24 = 23` ⇒ trễ đúng một ngày thì giờ đến vẫn `23`.
- `22 + 5 = 27` ⇒ `27 - 24 = 3` ⇒ đến lúc `3` giờ sáng hôm sau.

In [7]:
for a, d in [(15, 5), (13, 11), (23, 24), (22, 5)]:
    total = a + d
    print(f"{a} + {d} = {total}  ->  {total % 24}")

15 + 5 = 20  ->  20
13 + 11 = 24  ->  0
23 + 24 = 47  ->  23
22 + 5 = 27  ->  3


## 5. Cách 1 — Brute Force / baseline: mô phỏng từng giờ trôi qua

**Ý tưởng:** bắt đầu từ `t = arrivalTime`; lặp `delayedTime` lần, mỗi lần tăng `t` thêm `1` giờ và nếu `t` chạm `24` thì đặt lại về `0`. Đây đúng là cách đồng hồ hoạt động.

In [8]:
class SolutionSimulate:
    def findDelayedArrivalTime(self, arrivalTime: int, delayedTime: int) -> int:
        t = arrivalTime
        for _ in range(delayedTime):     # trôi qua từng giờ
            t += 1
            if t == 24:                  # chạm nửa đêm thì quay về 0
                t = 0
        return t

In [9]:
sol = SolutionSimulate()
print(sol.findDelayedArrivalTime(15, 5))      # 20
print(sol.findDelayedArrivalTime(13, 11))     # 0
print(sol.findDelayedArrivalTime(23, 24))     # 23

20
0
23


**Độ phức tạp:** Time `O(delayedTime)` (tối đa 24 vòng); Space `O(1)`.

**Nút thắt:** về mặt thực hành với `delayedTime <= 24` thì quá nhanh; nhưng nếu `delayedTime` lên tới `10^9` thì vòng lặp sẽ rất chậm. Ta muốn công thức **`O(1)`** không phụ thuộc độ lớn của `delayedTime`.

## 6. Ý tưởng thuật toán / bước đột phá

**Đột phá:** vì chỉ có 24 vị trí trên đồng hồ, **số dư khi chia cho 24** cho ngay vị trí cuối cùng mà không cần đếm từng bước:

`(arrivalTime + delayedTime) % 24`

Vì sao? Mỗi lần cộng đủ 24 giờ ta trở lại đúng chỗ cũ, nên ta chỉ cần quan tâm phần **dư** của tổng chia 24.

Với ràng buộc của đề (tổng `<= 47`), có thể thay `%` bằng một lần trừ `24` — nhưng `%` tổng quát và an toàn hơn.

In [10]:
a, d = 22, 5
total = a + d
print("tổng =", total, "| số ngày trôi qua =", total // 24, "| giờ trên đồng hồ =", total % 24)

tổng = 27 | số ngày trôi qua = 1 | giờ trên đồng hồ = 3


## 7. Các cách tiếp theo

### Cách 2 — `if`: trừ `24` khi tổng `>= 24`

Vì tổng `< 48`, tối đa quay vòng một lần: nếu `total >= 24` thì `total - 24`, ngược lại giữ nguyên. Chú ý dùng `>=` (không phải `>`), vì `total == 24` phải ra `0`.

In [11]:
class SolutionIf:
    def findDelayedArrivalTime(self, arrivalTime: int, delayedTime: int) -> int:
        total = arrivalTime + delayedTime
        if total >= 24:
            total -= 24
        return total

print(SolutionIf().findDelayedArrivalTime(15, 5))      # 20
print(SolutionIf().findDelayedArrivalTime(13, 11))     # 0
print(SolutionIf().findDelayedArrivalTime(23, 24))     # 23

20
0
23


**Độ phức tạp:** Time `O(1)`, Space `O(1)`. **Cẩn thận:** đúng **chỉ khi** tổng `< 48` như ràng buộc của đề. Nếu `delayedTime` lớn hơn, một lần trừ không đủ — phải dùng `%`.

### Cách 3 — Phép `%` (cách nộp)

`(arrivalTime + delayedTime) % 24` — một dòng, đúng với mọi `delayedTime` nguyên không âm.

In [12]:
class SolutionMod:
    def findDelayedArrivalTime(self, arrivalTime: int, delayedTime: int) -> int:
        return (arrivalTime + delayedTime) % 24

print(SolutionMod().findDelayedArrivalTime(15, 5))      # 20
print(SolutionMod().findDelayedArrivalTime(13, 11))     # 0
print(SolutionMod().findDelayedArrivalTime(23, 24))     # 23
print(SolutionMod().findDelayedArrivalTime(1, 10**9))   # đúng cả khi trễ cực lớn

20
0
23
17


**Độ phức tạp:** Time `O(1)`, Space `O(1)`.

### Cách 4 — Dùng thư viện `datetime`

Tạo một thời điểm bất kỳ có giờ là `arrivalTime`, cộng `timedelta(hours=delayedTime)`, rồi đọc `.hour`. Thư viện lo việc quay vòng ngày. Hữu ích khi đề có cả phút, ngày, tháng; ở bài này là hơi "dùng dao mổ trâu giết gà" nhưng chứng minh `%` cho kết quả như đồng hồ thật.

In [13]:
from datetime import datetime, timedelta

class SolutionDatetime:
    def findDelayedArrivalTime(self, arrivalTime: int, delayedTime: int) -> int:
        start = datetime(2000, 1, 1, arrivalTime)          # năm/tháng/ngày bất kỳ
        return (start + timedelta(hours=delayedTime)).hour

print(SolutionDatetime().findDelayedArrivalTime(15, 5))      # 20
print(SolutionDatetime().findDelayedArrivalTime(13, 11))     # 0
print(SolutionDatetime().findDelayedArrivalTime(23, 24))     # 23

20
0
23


**Độ phức tạp:** Time `O(1)`, Space `O(1)` (nhưng hằng số lớn hơn vì tạo đối tượng).

### Cách 5 — Xoay vòng với `deque.rotate`

Tạo `dq = deque(range(24))`, `rotate(-delayedTime)` để xoay trái; phần tử ở vị trí `arrivalTime` chính là `(arrivalTime + delayedTime) % 24`. Cách này **hình dung rõ** đồng hồ là vòng tròn xoay.

In [14]:
from collections import deque

class SolutionDeque:
    def findDelayedArrivalTime(self, arrivalTime: int, delayedTime: int) -> int:
        clock = deque(range(24))          # 0, 1, ..., 23 xếp theo vòng tròn
        clock.rotate(-delayedTime)        # xoay trái delayedTime bước
        return clock[arrivalTime]

print(SolutionDeque().findDelayedArrivalTime(15, 5))      # 20
print(SolutionDeque().findDelayedArrivalTime(13, 11))     # 0
print(SolutionDeque().findDelayedArrivalTime(23, 24))     # 23

20
0
23


**Độ phức tạp:** tạo `deque` `O(24)` và `rotate` `O(min(k, 24))` ⇒ coi như `O(1)` vì 24 là hằng số; Space `O(24)`. Chỉ có tác dụng minh hoạ.

## 8. TRACE từng bước

Trace Cách 1 (mô phỏng) với `arrivalTime = 22`, `delayedTime = 5` — ca có quay vòng qua nửa đêm:

| vòng | `t` trước | `t + 1` | chạm `24`? | `t` sau |
|---|---|---|---|---|
| 1 | 22 | 23 | không | 23 |
| 2 | 23 | 24 | **có** ⇒ đặt `0` | 0 |
| 3 | 0 | 1 | không | 1 |
| 4 | 1 | 2 | không | 2 |
| 5 | 2 | 3 | không | 3 |

Kết quả `3`, trùng với `(22 + 5) % 24 = 27 % 24 = 3`.

In [15]:
def trace_simulate(arrival, delayed):
    t = arrival
    print(f"bắt đầu: t = {t}")
    for k in range(1, delayed + 1):
        before = t
        t += 1
        note = ""
        if t == 24:
            t = 0
            note = "  <- chạm 24 nên quay về 0"
        print(f"vòng {k}: {before} -> {t}{note}")
    print("kết quả:", t)
    return t

trace_simulate(22, 5)
print("công thức :", (22 + 5) % 24)

bắt đầu: t = 22
vòng 1: 22 -> 23
vòng 2: 23 -> 0  <- chạm 24 nên quay về 0
vòng 3: 0 -> 1
vòng 4: 1 -> 2
vòng 5: 2 -> 3
kết quả: 3
công thức : 3


In [16]:
def trace_mod(arrival, delayed):
    total = arrival + delayed
    days, hour = divmod(total, 24)
    print(f"{arrival} + {delayed} = {total} = {days} x 24 + {hour}  =>  giờ {hour}")
    return hour

trace_mod(15, 5)
trace_mod(13, 11)
trace_mod(23, 24)

15 + 5 = 20 = 0 x 24 + 20  =>  giờ 20
13 + 11 = 24 = 1 x 24 + 0  =>  giờ 0
23 + 24 = 47 = 1 x 24 + 23  =>  giờ 23


23

## 9. Code nộp LeetCode

Dùng Cách 3: một dòng, đúng cho mọi `delayedTime`.

In [17]:
class Solution:
    def findDelayedArrivalTime(self, arrivalTime: int, delayedTime: int) -> int:
        return (arrivalTime + delayedTime) % 24

In [18]:
print(Solution().findDelayedArrivalTime(15, 5))      # 20
print(Solution().findDelayedArrivalTime(13, 11))     # 0

20
0


## 10. Vì sao thuật toán đúng

1. **Mô hình.** Giờ trên đồng hồ 24 giờ là số dư của tổng số giờ đã trôi chia cho 24. Cộng thêm một giờ là `t → (t + 1) % 24`. Làm `delayedTime` lần liên tiếp ta có `t → (t + delayedTime) % 24` (vì `((t + 1) % 24 + 1) % 24 = (t + 2) % 24` — số dư không bị đổi khi cộng thêm bội của 24).
2. **Giá trị nằm trong `0..23`.** Số dư chia cho `24` luôn thuộc `0..23` — đúng tập giá trị hợp lệ của giờ, không bao giờ trả `24`.
3. **Ví dụ biên.** `arrival + delayed = 24` cho số dư `0`: nửa đêm; `arrival + delayed = 47` cho `23`.
4. **Cách 2 (`if`)** tương đương `%` vì tổng nằm trong `2..47`: nếu tổng `< 24` thì dư là chính nó; nếu `24 <= tổng < 48` thì dư là `tổng - 24`. Hai nhánh này bao phủ toàn bộ miền của đề.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng từng giờ | `O(delayedTime)` | `O(1)` |
| 2. `if` trừ 24 | `O(1)` | `O(1)` |
| 3. `%` | `O(1)` | `O(1)` |
| 4. `datetime` | `O(1)` (hằng số lớn) | `O(1)` |
| 5. `deque.rotate` | `O(1)` (vì 24 cố định) | `O(1)` (24 phần tử) |

Với đề này mọi cách đều chạy tức thì. Điểm khác biệt chỉ là khi `delayedTime` rất lớn: Cách 1 chậm dần, các cách còn lại vẫn nhanh (Cách 2 sai nếu tổng `>= 48`).

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Các ca biên: giờ nhỏ nhất `arrival = 1`; giờ lớn nhất `23`; trễ nhỏ nhất `1`; trễ lớn nhất `24`; tổng đúng `24` (ra `0`); tổng đúng `47` (ra `23`); tổng `23` (không quay vòng).

In [19]:
sols = [SolutionSimulate(), SolutionIf(), SolutionMod(), SolutionDatetime(), SolutionDeque(), Solution()]
cases = [
    ("ví dụ 1", 15, 5, 20),
    ("ví dụ 2", 13, 11, 0),
    ("nhỏ nhất", 1, 1, 2),
    ("tổng 23 chưa quay vòng", 22, 1, 23),
    ("tổng 24 ra 0", 23, 1, 0),
    ("tổng 24 ra 0 (kiểu khác)", 1, 23, 0),
    ("trễ cả ngày", 5, 24, 5),
    ("lớn nhất", 23, 24, 23),
    ("23 + 2", 23, 2, 1),
    ("12 + 12", 12, 12, 0),
]
for name, a, d, exp in cases:
    for s in sols:
        got = s.findDelayedArrivalTime(a, d)
        assert got == exp, (name, type(s).__name__, got, exp)
    print("PASS", name, "->", exp)

PASS ví dụ 1 -> 20
PASS ví dụ 2 -> 0
PASS nhỏ nhất -> 2
PASS tổng 23 chưa quay vòng -> 23
PASS tổng 24 ra 0 -> 0
PASS tổng 24 ra 0 (kiểu khác) -> 0
PASS trễ cả ngày -> 5
PASS lớn nhất -> 23
PASS 23 + 2 -> 1
PASS 12 + 12 -> 0


### 12.2. Vét cạn **toàn bộ** đầu vào hợp lệ

Miền đầu vào chỉ có `23 × 24 = 552` cặp `(arrivalTime, delayedTime)`. Ta có thể kiểm tra **tất cả** và so với oracle mô phỏng từng giờ viết độc lập (dùng danh sách vòng tròn chứ không dùng `%`).

In [20]:
def oracle(arrival, delayed):
    hours = list(range(24))
    pos = arrival
    for _ in range(delayed):
        pos = (pos + 1) if pos + 1 < 24 else 0
    return hours[pos]

count = 0
for a in range(1, 24):
    for d in range(1, 25):
        exp = oracle(a, d)
        for s in sols:
            got = s.findDelayedArrivalTime(a, d)
            assert got == exp, (type(s).__name__, a, d, got, exp)
        assert 0 <= exp <= 23
        count += 1
print(count, "cặp hợp lệ: mọi cách đều khớp oracle")

552 cặp hợp lệ: mọi cách đều khớp oracle


### 12.3. Đối chiếu ngẫu nhiên có seed cho `delayedTime` lớn (ngoài đề)

Ngoài constraints, kiểm tra riêng các cách tổng quát (không dùng Cách 2 vì nó chỉ đúng khi tổng `< 48`).

In [21]:
import random
random.seed(2651)
general = [SolutionSimulate(), SolutionMod(), SolutionDatetime(), SolutionDeque()]
for _ in range(300):
    a = random.randint(1, 23)
    d = random.randint(1, 500)
    exp = (a + d) % 24
    for s in general:
        assert s.findDelayedArrivalTime(a, d) == exp, (type(s).__name__, a, d)
# chứng minh Cách 2 hỏng nếu vượt ràng buộc
assert SolutionIf().findDelayedArrivalTime(1, 100) != (1 + 100) % 24
print("300 ca delayedTime lớn khớp; Cách 2 chỉ đúng trong ràng buộc của đề")

300 ca delayedTime lớn khớp; Cách 2 chỉ đúng trong ràng buộc của đề


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên quay vòng (trả `24` hoặc lớn hơn)

In [22]:
def sai_khong_quay_vong(arrivalTime: int, delayedTime: int) -> int:
    return arrivalTime + delayedTime

print("sai :", sai_khong_quay_vong(13, 11))                 # 24
print("đúng:", Solution().findDelayedArrivalTime(13, 11))   # 0

sai : 24
đúng: 0


**Vì sao sai:** giờ phải thuộc `0..23`. `24` giờ chính là `0` giờ của ngày hôm sau. Phải có phép `% 24` (hoặc trừ `24`).

### Lỗi 2: Dùng `% 23` hoặc `% 25` (nhầm số vạch của đồng hồ)

In [23]:
def sai_mod_23(arrivalTime: int, delayedTime: int) -> int:
    return (arrivalTime + delayedTime) % 23

print("sai :", sai_mod_23(15, 5), sai_mod_23(23, 1))        # 20 1  (23+1 phải là 0)
print("đúng:", Solution().findDelayedArrivalTime(15, 5), Solution().findDelayedArrivalTime(23, 1))   # 20 0

sai : 20 1
đúng: 20 0


**Vì sao sai:** đồng hồ có **24** vạch `0..23`, nên chu kỳ là `24`. Chu kỳ `23` làm `23 + 1 = 24 % 23 = 1`, nhưng đáp án phải là `0`.

### Lỗi 3: Trừ `24` với điều kiện `>` (lệch một đơn vị)

In [24]:
def sai_lech_mot(arrivalTime: int, delayedTime: int) -> int:
    total = arrivalTime + delayedTime
    if total > 24:                 # SAI: total == 24 không bị trừ
        total -= 24
    return total

print("sai :", sai_lech_mot(13, 11))                        # 24
print("đúng:", Solution().findDelayedArrivalTime(13, 11))   # 0

sai : 24
đúng: 0


**Vì sao sai:** `total == 24` đúng là nửa đêm `0` giờ nên phải trừ. Điều kiện đúng là `total >= 24`. Đây chính là ví dụ 2 của đề!

### Lỗi 4: Dùng `//` thay cho `%`

In [25]:
def sai_chia_nguyen(arrivalTime: int, delayedTime: int) -> int:
    return (arrivalTime + delayedTime) // 24     # đây là số ngày, không phải giờ

print("sai :", sai_chia_nguyen(15, 5), sai_chia_nguyen(22, 5))   # 0 1
print("đúng:", Solution().findDelayedArrivalTime(15, 5), Solution().findDelayedArrivalTime(22, 5))   # 20 3

sai : 0 1
đúng: 20 3


**Vì sao sai:** `//` cho **thương** (số ngày đã qua), còn đề cần **số dư** (giờ còn lại). Nhớ: `%` là "phần lẻ" trên vòng tròn.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Mọi bài có tính chất **tuần hoàn / quay vòng** (giờ, ngày trong tuần, chỉ số mảng vòng, tháng) ⇒ nghĩ ngay tới `% chu_kỳ`.
- Công thức chung cho đồng hồ: `(x + k) % N`; lùi thì `(x - k) % N` (Python tự cho kết quả không âm).
- `divmod(a, b)` cho cả thương và dư; hữu ích khi cần "bao nhiêu ngày và bao nhiêu giờ".
- Kiểm tra biên: kết quả có bao giờ bằng `N` không? (Phải là `0`.)
- Nếu miền đầu vào nhỏ (như 23 × 24 = 552 cặp), có thể **vét cạn** mọi đầu vào để tự kiểm chứng công thức.

## 15. Biến thể của bài

1. **Đến sớm:** `(arrivalTime - earlyTime) % 24` — thử với số âm trong Python.
2. **Cộng giờ và phút:** đưa về phút `(h * 60 + m + delay) % 1440` rồi tách `divmod(.., 60)`.
3. **Ngày trong tuần sau `k` ngày:** `(day + k) % 7` (liên quan Day of the Week, LeetCode 1185).
4. **Mảng vòng:** truy cập `nums[(i + k) % n]`; bài **Rotate Array (LeetCode 189)** là cùng ý tưởng.
5. **Number of Days Between Two Dates (LeetCode 1360)** và các bài về thời gian: biến đổi sang đơn vị nhỏ nhất rồi dùng `divmod`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng từng giờ | Tăng `t`, chạm `24` thì về `0` | `O(delayedTime)` | `O(1)` | Hiểu bản chất đồng hồ |
| 2. `if` trừ 24 | Tối đa quay vòng một lần | `O(1)` | `O(1)` | Chỉ khi tổng `< 48` (đúng với đề này) |
| 3. `%` | Số dư của tổng chia 24 | `O(1)` | `O(1)` | **Nên nộp**, đúng tổng quát |
| 4. `datetime` | Thư viện thời gian | `O(1)` | `O(1)` | Khi đề có ngày/phút/múi giờ |
| 5. `deque.rotate` | Xoay vòng tròn 24 phần tử | `O(1)` | `O(1)` | Minh hoạ, không thực dụng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `(20 + 5) % 24`, `-1 % 24`, `divmod(50, 24)`, `(3 - 5) % 24`.

**Bài 2 (tính tay).** Tính `findDelayedArrivalTime(22, 7)` và `findDelayedArrivalTime(1, 24)`; viết rõ "tổng = ... ⇒ dư ...".

**Bài 3 (điền TODO).** Hoàn thành `delayed_time_todo(arrival, delayed)` bằng `if` (không dùng `%`).

In [26]:
def delayed_time_todo(arrival, delayed):
    total = arrival + delayed
    # TODO 1: nếu total >= 24 thì trừ 24 (chú ý >= chứ không phải >)
    # TODO 2: return total
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `delayed_time_deque(arrival, delayed)` bằng `deque.rotate` mà không nhìn Cách 5. Kiểm tra trên mọi cặp hợp lệ so với `%`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `early_time(arrival, early)` trả giờ khi tàu **đến sớm** `early` giờ (kết quả vẫn trong `0..23`, `early` có thể lớn hơn `arrival`). (b) Viết `add_minutes(h, m, delay_min)` trả `(giờ, phút)` sau khi cộng `delay_min` phút, trong định dạng 24 giờ.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `25 % 24 = 1`; Python cho `-1 % 24 = 23`; `50 = 2 × 24 + 2`; `3 - 5 = -2`, `-2 % 24 = 22`. Bài 2: `22 + 7 = 29` ⇒ `29 - 24 = 5`; `1 + 24 = 25` ⇒ `1`.
- **Mức 2 (vừa):** Bài 3: `if total >= 24: total -= 24`. Bài 4: `clock = deque(range(24)); clock.rotate(-delayed); return clock[arrival]`.
- **Mức 3 (sâu):** Bài 5(a): `(arrival - early) % 24` — dấu trừ và `%` của Python lo phần lùi qua nửa đêm. Bài 5(b): `total = h * 60 + m + delay_min`; rồi `total %= 1440`; `return divmod(total, 60)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert (20 + 5) % 24 == 1
assert -1 % 24 == 23
assert divmod(50, 24) == (2, 2)
assert (3 - 5) % 24 == 22

# Bài 2
assert Solution().findDelayedArrivalTime(22, 7) == 5
assert Solution().findDelayedArrivalTime(1, 24) == 1

# Bài 3
def delayed_time_ans(arrival, delayed):
    total = arrival + delayed
    if total >= 24:
        total -= 24
    return total

for a in range(1, 24):
    for d in range(1, 25):
        assert delayed_time_ans(a, d) == (a + d) % 24

# Bài 4
from collections import deque

def delayed_time_deque(arrival, delayed):
    clock = deque(range(24))
    clock.rotate(-delayed)
    return clock[arrival]

for a in range(1, 24):
    for d in range(1, 25):
        assert delayed_time_deque(a, d) == (a + d) % 24

# Bài 5
def early_time(arrival, early):
    return (arrival - early) % 24

assert early_time(3, 5) == 22
assert early_time(10, 3) == 7
assert early_time(5, 29) == 0

def add_minutes(h, m, delay_min):
    total = (h * 60 + m + delay_min) % 1440
    return divmod(total, 60)

assert add_minutes(23, 50, 20) == (0, 10)
assert add_minutes(10, 15, 45) == (11, 0)
assert add_minutes(0, 0, 1440) == (0, 0)
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đồng hồ vòng tròn: `(x + k) % N`)

```python
class Solution:
    def findDelayedArrivalTime(self, arrivalTime: int, delayedTime: int) -> int:
        return (arrivalTime + delayedTime) % 24
```

**Checklist**

- [ ] Kết quả luôn trong `0..23` (không bao giờ `24`)?
- [ ] Dùng `% 24` (không phải `% 23`, `% 25`, hay `// 24`)?
- [ ] Nếu dùng `if`: điều kiện là `total >= 24` (không phải `>`)?
- [ ] Đã thử `13 + 11 = 24` (ra `0`), `23 + 24 = 47` (ra `23`), `1 + 1`?
- [ ] Hiểu rằng `-1 % 24 == 23` trong Python nhưng có thể là `-1` ở C++/Java?
- [ ] Signature đúng `findDelayedArrivalTime(self, arrivalTime: int, delayedTime: int) -> int`?